In [1]:
import pandas as pd

team_name_mapping = {
    "Arizona Cardinals": "ARI",
    "Atlanta Falcons": "ATL",
    "Baltimore Ravens": "BAL",
    "Buffalo Bills": "BUF",
    "Carolina Panthers": "CAR",
    "Chicago Bears": "CHI",
    "Cincinnati Bengals": "CIN",
    "Cleveland Browns": "CLE",
    "Dallas Cowboys": "DAL",
    "Denver Broncos": "DEN",
    "Detroit Lions": "DET",
    "Green Bay Packers": "GB",
    "Houston Texans": "HOU",
    "Indianapolis Colts": "IND",
    "Jacksonville Jaguars": "JAX",
    "Kansas City Chiefs": "KC",
    "Miami Dolphins": "MIA",
    "Minnesota Vikings": "MIN",
    "New England Patriots": "NE",
    "New Orleans Saints": "NO",
    "New York Giants": "NYG",
    "New York Jets": "NYJ",
    "Oakland Raiders": "LV",  # Fixed
    "Philadelphia Eagles": "PHI",
    "Pittsburgh Steelers": "PIT",
    "San Diego Chargers": "LAC",
    "San Francisco 49ers": "SF",
    "Seattle Seahawks": "SEA",
    "St. Louis Rams": "LAR",  # Fixed
    "Tampa Bay Buccaneers": "TB",
    "Tennessee Titans": "TEN",
    "Washington Football Team": "WAS"  # Fixed
}

# Function to replace team names in the DataFrame
def replace_team_names(df, mapping):
    df['Team'] = df['Team'].map(mapping)
    return df

# File paths
files = {
    "Half PPR": "../PickleFiles/Half PPR Rankings.pkl",
    "Full PPR": "../PickleFiles/Full PPR Rankings.pkl",
    "Non PPR": "../PickleFiles/Non PPR Rankings.pkl"
}

# Define weights
vbd_weight = 0.3
adp_weight = 0.7

# Loop through each file
for key, file in files.items():
    # Load the pickle file
    df = pd.read_pickle(file)
    
    # Get the baseline points for each position
    baseline_qb = df[(df['Position'] == 'QB')]['Final PPG'].iloc[min(11, len(df[df['Position'] == 'QB'])-1)]
    baseline_rb = df[(df['Position'] == 'RB')]['Final PPG'].iloc[min(23, len(df[df['Position'] == 'RB'])-1)]
    baseline_wr = df[(df['Position'] == 'WR')]['Final PPG'].iloc[min(29, len(df[df['Position'] == 'WR'])-1)]
    baseline_te = df[(df['Position'] == 'TE')]['Final PPG'].iloc[min(11, len(df[df['Position'] == 'TE'])-1)]
    
    # Calculate VBD for each player based on their position
    def calculate_vbd(row):
        if row['Position'] == 'QB':
            return row['Final PPG'] - baseline_qb
        elif row['Position'] == 'RB':
            return row['Final PPG'] - baseline_rb
        elif row['Position'] == 'WR':
            return row['Final PPG'] - baseline_wr
        elif row['Position'] == 'TE':
            return row['Final PPG'] - baseline_te
        else:
            return 0
    
    df['VBD'] = df.apply(calculate_vbd, axis=1)
    
    # Normalize VBD and ESPN ADP
    df['Normalized VBD'] = (df['VBD'] - df['VBD'].min()) / (df['VBD'].max() - df['VBD'].min())
    df['Normalized ADP'] = (df['ESPN ADP'] - df['ESPN ADP'].min()) / (df['ESPN ADP'].max() - df['ESPN ADP'].min())
    
    # Calculate the weighted score
    df['Weighted Score'] = (vbd_weight * df['Normalized VBD']) + (adp_weight * (1 - df['Normalized ADP']))
    
    # Sort by weighted score in descending order to get the top players based on the weighted score
    df = df.sort_values(by='Weighted Score', ascending=False)
    
    # Adjust the ranking column
    df['Rank'] = range(1, len(df) + 1)

    df = replace_team_names(df, team_name_mapping)

    df['Position Rank'] = df.groupby('Position').cumcount() + 1
    df['Position'] = df['Position'] + df['Position Rank'].astype(str)
    df.drop(columns=['Position Rank'], inplace=True)
    
    # Save the updated DataFrame to a new pickle file
    output_file = file.replace(".pkl", " with Weighted VBD.pkl")
    df.to_pickle(output_file)

df

/Users/kmaran3/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


,Rank,Name,Team,Position,Final PPG,Bye Week,ESPN ADP,VBD,Normalized VBD,Normalized ADP,Weighted Score
0,1,C.J. Stroud,HOU,QB1,19.767020,14,NaN,3.426556,0.914037,NaN,NaN
1,2,Bryce Young,CAR,QB2,19.613560,11,NaN,3.273096,0.906367,NaN,NaN
2,3,Will Levis,TEN,QB3,19.112188,5,NaN,2.771724,0.881309,NaN,NaN
3,4,Tommy DeVito,NYG,QB4,18.993263,11,NaN,2.652798,0.875365,NaN,NaN
4,5,Skylar Thompson,MIA,QB5,18.685801,6,NaN,2.345337,0.859998,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
307,308,Joe Flacco,IND,QB49,2.016512,14,NaN,-14.323952,0.026879,NaN,NaN
308,309,Tyrod Taylor,NYJ,QB50,1.958345,12,NaN,-14.382119,0.023972,NaN,NaN
309,310,Tim Jones,JAX,WR109,1.878337,12,NaN,-2.436150,0.621023,NaN,NaN
310,311,Josh Johnson,BAL,QB51,1.856568,14,NaN,-14.483897,0.018886,NaN,NaN
